2. Projektni Zadatak: Razvoj Klasifikatora za Analizu
Sentimenta u PySpark-u

In [2]:
from pyspark.sql import SparkSession

sc = SparkSession.builder.appName("FinancialSentimentAnalysis").getOrCreate()


26/04/30 01:33:35 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
data = sc.read.csv("data.csv", header=True, inferSchema=True)
data.show()

+--------------------+---------+
|            Sentence|Sentiment|
+--------------------+---------+
|The GeoSolutions ...| positive|
|$ESI on lows, dow...| negative|
|For the last quar...| positive|
|According to the ...|  neutral|
|The Swedish buyou...|  neutral|
|$SPY wouldn't be ...| positive|
|Shell's $70 Billi...| negative|
|SSH COMMUNICATION...| negative|
|Kone 's net sales...| positive|
|The Stockmann dep...|  neutral|
|Circulation reven...| positive|
|$SAP Q1 disappoin...| negative|
|The subdivision m...| positive|
|Viking Line has c...|  neutral|
|Ahlstrom Corporat...|  neutral|
|$FB gone green on...| positive|
|$MSFT SQL Server ...| positive|
|According to L+ñn...|  neutral|
|The company 's sh...|  neutral|
|Elcoteq SE is lis...|  neutral|
+--------------------+---------+
only showing top 20 rows


In [ ]:
data.printSchema()

root
 |-- Sentence: string (nullable = true)
 |-- Sentiment: string (nullable = true)



In [6]:
from pyspark.sql.functions import count, when, col

data.select([count(when(col(c).isNull(), c)).alias(c) for c in data.columns]).show()

+--------+---------+
|Sentence|Sentiment|
+--------+---------+
|       0|        0|
+--------+---------+



In [7]:
data.groupBy("sentiment").count().show()

+--------------------+-----+
|           sentiment|count|
+--------------------+-----+
|            positive| 1852|
|             neutral| 3130|
|            negative|  859|
| the damage is do...|    1|
+--------------------+-----+



In [8]:
valid_labels = ["positive", "neutral", "negative"]

data = data.filter(col("Sentiment").isin(valid_labels))

data.groupBy("Sentiment").count().show()

+---------+-----+
|Sentiment|count|
+---------+-----+
| positive| 1852|
|  neutral| 3130|
| negative|  859|
+---------+-----+



In [9]:
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(
    inputCol="Sentiment",
    outputCol="label"
)

data = label_indexer.fit(data).transform(data)
data.select("Sentiment", "label").show(10)

+---------+-----+
|Sentiment|label|
+---------+-----+
| positive|  1.0|
| negative|  2.0|
| positive|  1.0|
|  neutral|  0.0|
|  neutral|  0.0|
| positive|  1.0|
| negative|  2.0|
| negative|  2.0|
| positive|  1.0|
|  neutral|  0.0|
+---------+-----+
only showing top 10 rows


In [10]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

print("Train:", train_data.count())
print("Test:", test_data.count())

Train: 4724
Test: 1117


In [14]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover

tokenizer = Tokenizer(
    inputCol="Sentence",
    outputCol="tokens"
)

stop_remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

In [15]:
import nltk
nltk.download('punkt')
from nltk.stem import PorterStemmer
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

stemmer = PorterStemmer()

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

stem_udf = udf(stem_tokens, ArrayType(StringType()))

[nltk_data] Downloading package punkt to /Users/vesco/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [16]:
tokenized_data = tokenizer.transform(train_data)
filtered_data = stop_remover.transform(tokenized_data)

stemmed_data = filtered_data.withColumn(
    "stemmed_tokens",
    stem_udf(col("filtered_tokens"))
)

stemmed_data.select("Sentence", "stemmed_tokens").show(5, truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------+
|Sentence                                                                                                                                      |stemmed_tokens                                                                                                                    |
+----------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------+
|"$SPY Less than 0.2% down and people are calling it bearish. Some heading for exits already. Maybe 1% down will be ""the crash""? Disturbing!"|["$spi, less, 0.2%, peopl, c

In [18]:
from pyspark.ml.feature import CountVectorizer, IDF

cv = CountVectorizer(
    inputCol="stemmed_tokens",
    outputCol="raw_features"
)

cv_model = cv.fit(stemmed_data)

featurized_data = cv_model.transform(stemmed_data)

idf = IDF(
    inputCol="raw_features",
    outputCol="features"
)

idf_model = idf.fit(featurized_data)

tfidf_data = idf_model.transform(featurized_data)
tfidf_data.select("features", "label").show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                                                                                                                                                                                                                                                             |label|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [19]:
tokenized_test = tokenizer.transform(test_data)
filtered_test = stop_remover.transform(tokenized_test)

stemmed_test = filtered_test.withColumn(
    "stemmed_tokens",
    stem_udf(col("filtered_tokens"))
)

test_featurized = cv_model.transform(stemmed_test)
test_tfidf = idf_model.transform(test_featurized)

In [20]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20
)

lr_model = lr.fit(tfidf_data)

predictions = lr_model.transform(test_tfidf)

predictions.select("label", "prediction").show(10)

accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

precision_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

accuracy = accuracy_eval.evaluate(predictions)
f1 = f1_eval.evaluate(predictions)
precision = precision_eval.evaluate(predictions)
recall = recall_eval.evaluate(predictions)

print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Precision:", precision)
print("Recall:", recall)

26/04/30 02:00:43 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


+-----+----------+
|label|prediction|
+-----+----------+
|  1.0|       1.0|
|  2.0|       1.0|
|  1.0|       1.0|
|  2.0|       2.0|
|  2.0|       1.0|
|  2.0|       1.0|
|  1.0|       1.0|
|  2.0|       1.0|
|  1.0|       1.0|
|  2.0|       1.0|
+-----+----------+
only showing top 10 rows
Accuracy: 0.6025067144136079
F1 Score: 0.6056803847583269
Precision: 0.6090462781231712
Recall: 0.6025067144136079


In [21]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def evaluate_model(predictions):
    metrics = {}
    
    for metric in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
        evaluator = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName=metric
        )
        metrics[metric] = evaluator.evaluate(predictions)
    
    return metrics

In [22]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier

def get_model(model_name, params={}):
    if model_name == "lr":
        return LogisticRegression(featuresCol="features", labelCol="label", **params)
    
    elif model_name == "dt":
        return DecisionTreeClassifier(featuresCol="features", labelCol="label", **params)
    
    elif model_name == "rf":
        return RandomForestClassifier(featuresCol="features", labelCol="label", **params)
    
    else:
        raise ValueError("Unknown model")

In [23]:
def run_tfidf_experiment(train_data, test_data, model_name, params={}):
    
    # preprocessing train
    tokenized_train = tokenizer.transform(train_data)
    filtered_train = stop_remover.transform(tokenized_train)
    stemmed_train = filtered_train.withColumn("stemmed_tokens", stem_udf(col("filtered_tokens")))
    
    # TF-IDF train
    cv = CountVectorizer(inputCol="stemmed_tokens", outputCol="raw_features")
    cv_model = cv.fit(stemmed_train)
    train_featurized = cv_model.transform(stemmed_train)

    idf = IDF(inputCol="raw_features", outputCol="features")
    idf_model = idf.fit(train_featurized)
    train_final = idf_model.transform(train_featurized)

    # preprocessing test
    tokenized_test = tokenizer.transform(test_data)
    filtered_test = stop_remover.transform(tokenized_test)
    stemmed_test = filtered_test.withColumn("stemmed_tokens", stem_udf(col("filtered_tokens")))

    test_featurized = cv_model.transform(stemmed_test)
    test_final = idf_model.transform(test_featurized)

    # model
    model = get_model(model_name, params)
    trained_model = model.fit(train_final)

    predictions = trained_model.transform(test_final)

    metrics = evaluate_model(predictions)

    return {
        "model": trained_model,
        "metrics": metrics
    }

In [24]:
result = run_tfidf_experiment(train_data, test_data, "lr", {"maxIter": 20})

print(result["metrics"])

{'accuracy': 0.6025067144136079, 'f1': 0.6056803847583269, 'weightedPrecision': 0.6090462781231712, 'weightedRecall': 0.6025067144136079}


In [25]:
def preprocess_data(df):
    tokenized = tokenizer.transform(df)
    filtered = stop_remover.transform(tokenized)

    stemmed = filtered.withColumn(
        "stemmed_tokens",
        stem_udf(col("filtered_tokens"))
    )

    return stemmed

In [26]:
def build_tfidf_features(train_df, test_df):
    cv = CountVectorizer(
        inputCol="stemmed_tokens",
        outputCol="raw_features"
    )
    cv_model = cv.fit(train_df)

    train_featurized = cv_model.transform(train_df)
    test_featurized = cv_model.transform(test_df)

    idf = IDF(
        inputCol="raw_features",
        outputCol="features"
    )
    idf_model = idf.fit(train_featurized)

    train_final = idf_model.transform(train_featurized)
    test_final = idf_model.transform(test_featurized)

    return train_final, test_final

In [27]:
from pyspark.ml.feature import NGram

def build_bigram_features(train_df, test_df):
    ngram = NGram(
        n=2,
        inputCol="stemmed_tokens",
        outputCol="bigrams"
    )

    train_ngram = ngram.transform(train_df)
    test_ngram = ngram.transform(test_df)

    cv = CountVectorizer(
        inputCol="bigrams",
        outputCol="raw_features"
    )
    cv_model = cv.fit(train_ngram)

    train_featurized = cv_model.transform(train_ngram)
    test_featurized = cv_model.transform(test_ngram)

    idf = IDF(
        inputCol="raw_features",
        outputCol="features"
    )
    idf_model = idf.fit(train_featurized)

    train_final = idf_model.transform(train_featurized)
    test_final = idf_model.transform(test_featurized)

    return train_final, test_final

In [28]:
from pyspark.ml.feature import Word2Vec

def build_word2vec_features(train_df, test_df):
    word2vec = Word2Vec(
        vectorSize=100,
        minCount=1,
        inputCol="stemmed_tokens",
        outputCol="features"
    )

    w2v_model = word2vec.fit(train_df)

    train_final = w2v_model.transform(train_df)
    test_final = w2v_model.transform(test_df)

    return train_final, test_final

In [29]:
def run_experiment(train_data, test_data, feature_method, model_name, params={}):
    
    train_processed = preprocess_data(train_data)
    test_processed = preprocess_data(test_data)

    if feature_method == "tfidf":
        train_final, test_final = build_tfidf_features(train_processed, test_processed)

    elif feature_method == "bigram_tfidf":
        train_final, test_final = build_bigram_features(train_processed, test_processed)

    elif feature_method == "word2vec":
        train_final, test_final = build_word2vec_features(train_processed, test_processed)

    else:
        raise ValueError("Unknown feature method")

    model = get_model(model_name, params)
    trained_model = model.fit(train_final)

    predictions = trained_model.transform(test_final)

    metrics = evaluate_model(predictions)

    return {
        "feature_method": feature_method,
        "model_name": model_name,
        "params": params,
        "metrics": metrics,
        "model": trained_model
    }

In [30]:
def run_all_experiments(train_data, test_data):
    results = {}

    feature_methods = ["tfidf", "bigram_tfidf", "word2vec"]
    models = ["lr", "dt", "rf"]

    for feature_method in feature_methods:
        for model_name in models:
            key = f"{feature_method}_{model_name}"

            print(f"Running {key}...")

            result = run_experiment(
                train_data,
                test_data,
                feature_method,
                model_name
            )

            results[key] = result["metrics"]

    return results

In [31]:
all_results = run_all_experiments(train_data, test_data)

for name, metrics in all_results.items():
    print(name, metrics)

Running tfidf_lr...
Running tfidf_dt...


Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


Running tfidf_rf...


Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


Running bigram_tfidf_lr...


...

Running word2vec_lr...
Running word2vec_dt...


Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


Running word2vec_rf...


Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


tfidf_lr {'accuracy': 0.5935541629364369, 'f1': 0.5996468734514094, 'weightedPrecision': 0.606394178101465, 'weightedRecall': 0.5935541629364369}
tfidf_dt {'accuracy': 0.6911369740376008, 'f1': 0.6385736977050701, 'weightedPrecision': 0.679941750370633, 'weightedRecall': 0.6911369740376007}
tfidf_rf {'accuracy': 0.5711727842435094, 'f1': 0.4162365538384855, 'weightedPrecision': 0.6155705406506804, 'weightedRecall': 0.5711727842435094}
bigram_tfidf_lr {'accuracy': 0.5577439570277529, 'f1': 0.5458033246067892, 'weightedPrecision': 0.5517339209649726, 'weightedRecall': 0.5577439570277529}
bigram_tfidf_dt {'accuracy': 0.5890778871978514, 'f1': 0.4552091436518191, 'weightedPrecision': 0.7611702683805031, 'weightedRecall': 0.5890778871978514}
bigram_tfidf_rf {'accuracy': 0.5702775290957923, 'f1': 0.4142152634367385, 'weightedPrecision': 0.3252164601916023, 'weightedRecall': 0.5702775290957923}
word2vec_lr {'accuracy': 0.6598030438675022, 'f1': 0.6309346337686254, 'weightedPrecision': 0.63360

In [32]:
param_grid = {
    "lr": [
        {"maxIter": 20},
        {"maxIter": 50}
    ],
    "dt": [
        {"maxDepth": 5},
        {"maxDepth": 10}
    ],
    "rf": [
        {"numTrees": 20},
        {"numTrees": 50}
    ]
}

In [33]:
def run_all_experiments(train_data, test_data, param_grid):
    results = {}

    feature_methods = ["tfidf", "bigram_tfidf", "word2vec"]

    for feature_method in feature_methods:
        for model_name, param_list in param_grid.items():
            for params in param_list:

                key = f"{feature_method}_{model_name}_{params}"

                print(f"Running {key}...")

                result = run_experiment(
                    train_data,
                    test_data,
                    feature_method,
                    model_name,
                    params
                )

                results[key] = result["metrics"]

    return results

In [34]:
all_results = run_all_experiments(train_data, test_data, param_grid)

Running tfidf_lr_{'maxIter': 20}...
Running tfidf_lr_{'maxIter': 50}...
Running tfidf_dt_{'maxDepth': 5}...


Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


Running tfidf_dt_{'maxDepth': 10}...


Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


Running tfidf_rf_{'numTrees': 20}...


...

Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


Running word2vec_rf_{'numTrees': 20}...


Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


Running word2vec_rf_{'numTrees': 50}...


Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [35]:
best_model = max(all_results.items(), key=lambda x: x[1]["f1"])

print("Best configuration:")
print(best_model[0])
print(best_model[1])

Best configuration:
tfidf_dt_{'maxDepth': 10}
{'accuracy': 0.7081468218442256, 'f1': 0.6619221702227189, 'weightedPrecision': 0.6837594132017342, 'weightedRecall': 0.7081468218442257}


In [36]:
import pandas as pd

results_table = []

for name, metrics in all_results.items():
    row = {"experiment": name}
    row.update(metrics)
    results_table.append(row)

results_df = pd.DataFrame(results_table)
results_df.sort_values(by="f1", ascending=False)

,experiment,accuracy,f1,weightedPrecision,weightedRecall
3,tfidf_dt_{'maxDepth': 10},0.708147,0.661922,0.683759,0.708147
2,tfidf_dt_{'maxDepth': 5},0.691137,0.638574,0.679942,0.691137
0,tfidf_lr_{'maxIter': 20},0.602507,0.605680,0.609046,0.602507
13,word2vec_lr_{'maxIter': 50},0.643688,0.605005,0.599859,0.643688
1,tfidf_lr_{'maxIter': 50},0.595345,0.599420,0.603785,0.595345
16,word2vec_rf_{'numTrees': 20},0.658908,0.594571,0.556044,0.658908
12,word2vec_lr_{'maxIter': 20},0.632945,0.591604,0.580863,0.632945
14,word2vec_dt_{'maxDepth': 5},0.620412,0.588193,0.591361,0.620412
17,word2vec_rf_{'numTrees': 50},0.647269,0.578575,0.543538,0.647269
15,word2vec_dt_{'maxDepth': 10},0.578335,0.565852,0.555943,0.578335


### Interpretacija rezultata

Na osnovu sprovedenih eksperimenata može se zaključiti da je najbolji model za analizu sentimenta na Financial Sentiment Analysis skupu podataka bio **Decision Tree** sa **TF-IDF** reprezentacijom teksta i hiperparametrom **maxDepth = 10**.

Ovaj model je ostvario:

- Accuracy: 0.708
- F1 score: 0.662
- Weighted Precision: 0.684
- Weighted Recall: 0.708

Rezultati pokazuju da je **TF-IDF** reprezentacija bila efikasnija od **Word2Vec** i **TF-IDF sa bigramima**, što ukazuje da su pojedinačne reči bile dovoljno informativne za klasifikaciju sentimenta finansijskih vesti.

Modeli bazirani na **Word2Vec** ostvarili su solidne rezultate, ali nešto slabije od TF-IDF pristupa, verovatno zbog gubitka specifičnih informacija važnih za sentiment.

**TF-IDF sa bigramima** dao je slabije performanse, što može biti posledica povećane dimenzionalnosti i relativno malog broja uzoraka.

Najslabije rezultate dao je **Random Forest**, posebno na sparse TF-IDF reprezentacijama, što je očekivano zbog velike dimenzionalnosti ulaznog prostora.

### Zaključak

Na osnovu dobijenih rezultata preporučuje se upotreba **TF-IDF reprezentacije teksta** u kombinaciji sa **Decision Tree klasifikatorom**, jer je ovaj pristup dao najbolje rezultate na posmatranom skupu podataka.

Ovakav pristup je pogodan za praktične primene analize sentimenta u finansijskom domenu, gde su ključne reči i termini od velike važnosti za određivanje sentimenta.